In [1]:
import json
import os
import re
import urllib.request

sample_notes = """Website revamp kickoff
Alex: What are the main features you want on the new landing page
Client: We need a modern hero section, customer testimonials, and
Alex: Would you prefer Calendly integration or a custom-built sol
Client: Calendly works fine and keeps it simple.
Alex: Perfect, I ll prepare the initial wireframes by Wednesday."""

def offline_extract(notes):
    """Extract lines in the format Name: task by deadline."""
    actions = []
    # Matches "Name: ... by Deadline" at the end of a line
    pattern = re.compile(r"^([A-Za-z]+):\s*(.+?)\s+by\s+([A-Za-z0-9\s]+)\.?$", re.IGNORECASE)
    
    for line in notes.strip().splitlines():
        match = pattern.match(line.strip())
        if match:
            actions.append({
                "assignee": match.group(1),
                "task": match.group(2).strip(),
                "deadline": match.group(3).strip()
            })
    return actions

if _name_ == "_main_":
    extracted_actions = offline_extract(sample_notes)
    print(json.dumps(extracted_actions, indent=2))


<class 'NameError'>: name '_name_' is not defined

In [2]:
import json
import os
import re
import urllib.request

sample_notes = """Website revamp kickoff

Alex: What are the main features you want on the new landing page?
Client: We need a modern hero section, customer testimonials, and a contact form.
Alex: Would you prefer Calendly integration or a custom-built solution?
Client: Calendly works fine and keeps it simple.
Alex: Perfect, I'll prepare the initial wireframes by Wednesday.
"""

def offline_extract(notes):
    """Extract lines in the format Name: task by deadline."""

    actions = []

    pattern = r"(\w+):\s*(.*?)\s+by\s+([A-Za-z]+)"

    for line in notes.splitlines():

        match = re.search(pattern, line, re.IGNORECASE)

        if match:
            name = match.group(1)
            task = match.group(2)
            deadline = match.group(3)

            actions.append({
                "name": name,
                "task": task,
                "deadline": deadline
            })

    return actions


# Extract action items
result = offline_extract(sample_notes)

# Convert into JSON format
json_output = json.dumps(result, indent=4)

print("Extracted Action Items:")
print(json_output)

# Save results into a JSON file
with open("action_items.json", "w") as file:
    json.dump(result, file, indent=4)

print("\nData saved successfully!")

Extracted Action Items:
[
    {
        "name": "Alex",
        "task": "Perfect, I'll prepare the initial wireframes",
        "deadline": "Wednesday"
    }
]

Data saved successfully!


In [3]:
import json
import os
import re
import urllib.request
import urllib.error

sample_notes = """Website revamp kickoff
Alex: What are the main features you want on the new landing page?
Client: We need a modern hero section, customer testimonials, and a booking form.
Alex: Would you prefer Calendly integration or a custom-built solution?
Client: Calendly works fine and keeps it simple.
Alex: Perfect, I'll prepare the initial wireframes by Wednesday.
Client: Great, I'll send over the brand logo and testimonials by Friday."""

def offline_extract(notes):
    """Fallback: Extract lines in the format Name: task by deadline using regex."""
    actions = []
    pattern = re.compile(r'^([A-Za-z]+):\s*(.+?\s+by\s+[^.\n]+)', re.MULTILINE)
    
    for match in pattern.finditer(notes):
        name = match.group(1).strip()
        task_and_deadline = match.group(2).strip()
        
        # Clean up conversational filler
        task_and_deadline = re.sub(
            r"^(?:Great,\s*|Perfect,\s*|Sure,\s*)?(?:I'll|I will|we'll|we will)\s+",
            "",
            task_and_deadline,
            flags=re.IGNORECASE
        )
        actions.append(f"{name}: {task_and_deadline}")
        
    return actions

def llm_extract(notes, api_key):
    """Extract action items using an LLM API (OpenAI-compatible endpoint)."""
    url = "https://openai.com"
    prompt = (
        "Extract all action items from the following meeting notes. "
        "Format each item strictly as 'Name: task by deadline'. "
        "Return ONLY a JSON array of strings.\n\n"
        f"Notes:\n{notes}"
    )
    
    payload = {
        "model": "gpt-4o-mini",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }
    
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }
    )
    
    with urllib.request.urlopen(req, timeout=15) as response:
        result = json.loads(response.read().decode("utf-8"))
        content = result["choices"][0]["message"]["content"].strip()
        # Strip markdown code fences if present
        content = re.sub(r"^json\s*|\s*$", "", content)
        return json.loads(content)

def extract_action_items(notes):
    """Use LLM if OPENAI_API_KEY is available, otherwise fall back to offline regex."""
    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        try:
            print("Using LLM extractor...")
            return llm_extract(notes, api_key)
        except Exception as e:
            print(f"API call failed ({e}), switching to offline mode...")
            
    print("Using offline regex extractor...")
    return offline_extract(notes)

if _name_ == "_main_":
    action_items = extract_action_items(sample_notes)
    print("\nExtracted Action Items:")
    for item in action_items:
        print("-", item)

<class 'NameError'>: name '_name_' is not defined

In [4]:
import json
import os
import re
import urllib.request

sample_notes = """Website revamp kickoff
Alex: What are the main features you want on the new landing page?
Client: We need a modern hero section, customer testimonials, and a contact form.
Alex: Would you prefer Calendly integration or a custom-built solution?
Client: Calendly works fine and keeps it simple.
Alex: Perfect, I'll prepare the initial wireframes by Wednesday."""


def offline_extract(notes):
  """Extract lines in the format Name: task by deadline."""
  actions = []
  pattern = r"^([^:\n]+):\s*(.+?)\s+by\s+([^\n.]+)\.?"

  for line in notes.splitlines():
    match = re.match(pattern, line.strip())
    if match:
      actions.append({
          "owner": match.group(1).strip(),
          "task": match.group(2).strip(),
          "deadline": match.group(3).strip(),
      })
  return actions


def extract_actions(notes):
  """Uses an API if configured, otherwise falls back to offline_extract."""
  api_url = os.getenv("LLM_API_URL")
  api_key = os.getenv("LLM_API_KEY")

  if api_url and api_key:
    payload = json.dumps({"notes": notes}).encode("utf-8")
    req = urllib.request.Request(
        api_url,
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
        },
    )
    with urllib.request.urlopen(req) as response:
      return json.loads(response.read().decode("utf-8"))

  return offline_extract(notes)


print("App functions ready")

App functions ready


In [ ]:
%pip install ipywidgets

In [5]:
import ipywidgets as widgets

notes_box = widgets.Textarea(
    value=sample_notes,
    description="Notes:",
    layout=widgets.Layout(width="95%", height="170px"),
    style={"description_width": "initial"},
)

mode_box = widgets.ToggleButtons(
    options=[("Offline", "offline"), ("GEMINI FLASH 3.0", "live")],
    value="offline",
    description="Mode:",
)

button = widgets.Button(
    description="Extract action items", 
    button_style="primary", 
    icon="check"
)

output = widgets.Output()

def on_extract_clicked(_):
    with output:
        output.clear_output()
        text = notes_box.value.strip()
        if not text:
            print("Please enter meeting notes first.")
            return

        print("Mode:", "OFFLINE" if mode_box.value == "offline" else "LIVE GEMINI API")
        try:
            actions = offline_extract(text) if mode_box.value == "offline" else live_extract(text)
        except Exception as exc:
            # Keep raw provider errors and credentials off the shared screen
            print(f"Could not extract items ({type(exc).__name__}). Check setup and try again.")
            return

        if not actions:
            print("No actions found. In offline mode, use: Name: task by deadline")
            return

        for i, action in enumerate(actions, 1):
            print(f"\n{i}. {action['task']}")
            print(f"   Owner: {action['owner']} | Due: {action['deadline']}")

        print("\nReview every item against the original notes before using it.")

button.on_click(on_extract_clicked)
display(widgets.VBox([notes_box, mode_box, button, output]))

<class 'ModuleNotFoundError'>: No module named 'ipywidgets'

In [1]:
import ipywidgets as widgets

notes_box = widgets.Textarea(
    value=sample_notes,
    description="Notes:",
    layout=widgets.Layout(width="95%", height="170px"),
    style={"description_width": "initial"},
)

mode_box = widgets.ToggleButtons(
    options=[("Offline", "offline"), ("GEMINI FLASH 3.0", "live")],
    value="offline",
    description="Mode:",
)

button = widgets.Button(
    description="Extract action items", 
    button_style="primary", 
    icon="check"
)

output = widgets.Output()

def on_extract_clicked(_):
    with output:
        output.clear_output()
        text = notes_box.value.strip()
        if not text:
            print("Please enter meeting notes first.")
            return

        print("Mode:", "OFFLINE" if mode_box.value == "offline" else "LIVE GEMINI API")
        try:
            actions = offline_extract(text) if mode_box.value == "offline" else live_extract(text)
        except Exception as exc:
            # Keep raw provider errors and credentials off the shared screen
            print(f"Could not extract items ({type(exc).__name__}). Check setup and try again.")
            return

        if not actions:
            print("No actions found. In offline mode, use: Name: task by deadline")
            return

        for i, action in enumerate(actions, 1):
            print(f"\n{i}. {action['task']}")
            print(f"   Owner: {action['owner']} | Due: {action['deadline']}")

        print("\nReview every item against the original notes before using it.")

button.on_click(on_extract_clicked)
display(widgets.VBox([notes_box, mode_box, button, output]))

<class 'ModuleNotFoundError'>: No module named 'ipywidgets'

In [1]:
import ipywidgets as widgets

notes_box = widgets.Textarea(
    value=sample_notes,
    description="Notes:",
    layout=widgets.Layout(width="95%", height="170px"),
    style={"description_width": "initial"},
)

mode_box = widgets.ToggleButtons(
    options=[("Offline", "offline"), ("GEMINI FLASH 3.0", "live")],
    value="offline",
    description="Mode:",
)

button = widgets.Button(
    description="Extract action items", 
    button_style="primary", 
    icon="check"
)

output = widgets.Output()

def on_extract_clicked(_):
    with output:
        output.clear_output()
        text = notes_box.value.strip()
        if not text:
            print("Please enter meeting notes first.")
            return

        print("Mode:", "OFFLINE" if mode_box.value == "offline" else "LIVE GEMINI API")
        try:
            actions = offline_extract(text) if mode_box.value == "offline" else live_extract(text)
        except Exception as exc:
            # Keep raw provider errors and credentials off the shared screen
            print(f"Could not extract items ({type(exc).__name__}). Check setup and try again.")
            return

        if not actions:
            print("No actions found. In offline mode, use: Name: task by deadline")
            return

        for i, action in enumerate(actions, 1):
            print(f"\n{i}. {action['task']}")
            print(f"   Owner: {action['owner']} | Due: {action['deadline']}")

        print("\nReview every item against the original notes before using it.")

button.on_click(on_extract_clicked)
display(widgets.VBox([notes_box, mode_box, button, output]))

<class 'ModuleNotFoundError'>: No module named 'ipywidgets'

In [ ]:
%pip install ipywidgets

In [2]:

import ipywidgets as widgets
from IPython.display import display

# Sample meeting notes
sample_notes = """
Ravi: Complete the project report by Friday.
Priya: Prepare the presentation by Monday.
Rahul: Submit the assignment by Wednesday.
"""

# Offline extraction function
def offline_extract(text):
    actions = []

    for line in text.splitlines():
        if ":" in line:
            owner, task = line.split(":", 1)

            if " by " in task:
                task_name, deadline = task.split(" by ", 1)
            else:
                task_name = task
                deadline = "Not specified"

            actions.append({
                "owner": owner.strip(),
                "task": task_name.strip(),
                "deadline": deadline.strip()
            })

    return actions


# Notes input box
notes_box = widgets.Textarea(
    value=sample_notes,
    description="Notes:",
    layout=widgets.Layout(width="95%", height="170px"),
    style={"description_width": "initial"}
)

# Mode selection
mode_box = widgets.ToggleButtons(
    options=[
        ("Offline", "offline"),
        ("Gemini Flash 3.0", "live")
    ],
    value="offline",
    description="Mode:"
)

# Extract button
button = widgets.Button(
    description="Extract action items",
    button_style="primary",
    icon="check"
)

# Output area
output = widgets.Output()


# Button click function
def on_extract_clicked(_):
    with output:
        output.clear_output()

        text = notes_box.value.strip()

        if not text:
            print("Please enter meeting notes first.")
            return

        if mode_box.value == "live":
            print("Live Gemini API is not configured yet.")
            return

        print("Mode: OFFLINE")

        try:
            actions = offline_extract(text)

        except Exception as exc:
            print("Error:", type(exc).__name__)
            return

        if not actions:
            print("No actions found.")
            return

        for i, action in enumerate(actions, 1):
            print(f"\n{i}. {action['task']}")
            print(f"   Owner: {action['owner']}")
            print(f"   Due: {action['deadline']}")

        print("\nReview every item against the original notes.")


# Connect button
button.on_click(on_extract_clicked)

# Display widgets
display(
    widgets.VBox([
        notes_box,
        mode_box,
        button,
        output
    ])
)


<class 'ModuleNotFoundError'>: No module named 'ipywidgets'

In [3]:
import json
import os
import re
import urllib.request

sample_notes = """Website revamp kickoff

Alex: What are the main features you want on the new landing page?
Client: We need a modern hero section, customer testimonials, and a contact form.
Alex: Would you prefer Calendly integration or a custom-built solution?
Client: Calendly works fine and keeps it simple.
Alex: Perfect, I'll prepare the initial wireframes by Wednesday.
"""

def offline_extract(notes):
    """Extract lines in the format Name: task by deadline."""

    actions = []

    pattern = r"(\w+):\s*(.*?)\s+by\s+([A-Za-z]+)"

    for line in notes.splitlines():

        match = re.search(pattern, line, re.IGNORECASE)

        if match:
            name = match.group(1)
            task = match.group(2)
            deadline = match.group(3)

            actions.append({
                "name": name,
                "task": task,
                "deadline": deadline
            })

    return actions


# Extract action items
result = offline_extract(sample_notes)

# Convert into JSON format
json_output = json.dumps(result, indent=4)

print("Extracted Action Items:")
print(json_output)

# Save results into a JSON file
with open("action_items.json", "w") as file:
    json.dump(result, file, indent=4)

print("\nData saved successfully!")

Extracted Action Items:
[
    {
        "name": "Alex",
        "task": "Perfect, I'll prepare the initial wireframes",
        "deadline": "Wednesday"
    }
]

Data saved successfully!


In [4]:
%pip install ipywidgets